# PDF Loader가 중요한 이유
- Rag에서 PDF는 가장 흔한 데이터 소스 중 하나이며, PDF의 텍스트 레이아웃, 표 구조, 이미지 포함 여부에 따라 정확한 자료 추출 품질이 크게 달라진다. 
- 따라서 문서 형태에 맞는 로더 선택은 RAG 전체 품질을 좌우하는 핵심 요소다 

# PDF Loader 3종 핵심 비교 

### PyPDFLoader
- 가장 기본적인 텍스트 중심 PDF 로더 
- 가볍고 빠르지만 기능 제한 
- 단순 텍스트 문서에 적합 

### PDFPlumberLoader
- 표(Table)와 레이아웃 분석에 최적화된 로더
- 테이블 추출 정확도 최고 
- 형식이 있는 보고서나 논문에 강함 

### PyMuPDF4LLMLoader
- 이미지, 텍스트, 구조를 가장 유연하게 처리하는 최신 로더
- 텍스트 + 이미지 + 구조까지 모두 지원 
- PDF 내부 이미지 완전 추출 

In [3]:
pdf_attention = "data/Attention Is All You Need.pdf"
pdf_bert = "data/BERT.pdf"
pdf_lg_aimers = "data/LG Aimers 4기 소개자료.pdf"

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    file_path=pdf_lg_aimers, # 주소 
    extract_images=False) # 이미지 추출 여부 

In [6]:
docs = loader.load()
print(f"PDF 파일의 페이지 수: {len(docs)}")

PDF 파일의 페이지 수: 7


In [7]:
docs[0].metadata # 메타 정보 추출

{'producer': 'PyPDF',
 'creator': 'PyPDF',
 'creationdate': '2025-07-15T07:40:32+00:00',
 'moddate': '2025-07-15T07:40:32+00:00',
 'source': 'data/LG Aimers 4기 소개자료.pdf',
 'total_pages': 7,
 'page': 0,
 'page_label': '1'}

In [8]:
# 텍스트 추출 
print(docs[0].page_content)

LG AI연구원｜
LG Aimers 소개 자료


In [9]:
from pypdf import PdfReader

# PDF 파일 열기
reader = PdfReader(pdf_attention)

# 메타데이터 가져오기
metadata = reader.metadata

# 출력
print("PDF 메타데이터:")
for key, value in metadata.items():
    print(f"{key}: {value}")

PDF 메타데이터:
/Author: 
/CreationDate: D:20240410211143Z
/Creator: LaTeX with hyperref
/Keywords: 
/ModDate: D:20240410211143Z
/PTEX.Fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
/Producer: pdfTeX-1.40.25
/Subject: 
/Title: 
/Trapped: /False


In [10]:
for i in reader.pages:
    print("="*50)
    # 페이지별 텍스트 추출 
    print(i.extract_text()[:300])
    break 

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [11]:
from langchain_core.documents import Document

###########################################
# 초기화 
###########################################
dict_metadata = dict(reader.metadata)
documents = []
count = 0

###########################################
# 페이지별 처리  
###########################################
for i, page in enumerate(reader.pages):
    # 메타이데이터 추가 
    dict_metadata['page_no'] = i+1

    ###########################################
    # 페이지별 이미지 추가   
    ###########################################
    dict_metadata['lst_image_file_names'] = []
    for image_file_object in page.images:

        # Open and write the image data
        image_names = image_file_object.name.split(".")
        image_file_name = f"{image_names[0]}_{str(count)}.{image_names[1]}"
        with open(image_file_name, "wb") as fp:
            fp.write(image_file_object.data)
            count += 1
            dict_metadata['lst_image_file_names'].append(image_file_name)

    ###########################################
    # 페이지별 document 추가   
    ###########################################
    documents.append(Document(
        page_content=page.extract_text(),   # 페이지별 텍스트 
        metadata=dict_metadata              # 페이지별 메타데이터
    ))

In [12]:
documents[2].metadata

{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False',
 'page_no': 3,
 'lst_image_file_names': ['Im1_0.png']}

In [13]:
print(documents[2].page_content[:300]) # 추출된 텍스트 확인

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder
